In [0]:
# Install necessary packages
%pip install newspaper3k
%pip install nltk
%pip install lxml_html_clean==0.1.1
%pip install spacy

# Install the spaCy model
!python -m spacy download en_core_web_sm

# Import necessary libraries
from newspaper import Article, Config
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import logging
import spacy
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

# Download the missing NLTK resource
nltk.download('vader_lexicon')

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load the spaCy English model
nlp = spacy.load("en_core_web_sm")

# Load extracted data from CSV using pandas
dataframe = spark.read.format("delta").load("/mnt/data/extracted_news").toPandas()

# Function to fetch the full content of an article
def full_content(url):
    user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.102 Safari/537.36'
    config = Config()
    config.browser_user_agent = user_agent
    page = Article(url, config=config)

    try:
        page.download()
        page.parse()
        return page.text
    except Exception as e:
        logger.error(f"Error retrieving content from {url}: {e}")
        return 'couldnt retrieve'

# Function to fetch content in parallel for multiple URLs
def fetch_content_parallel(urls):
    with ThreadPoolExecutor() as executor:
        return list(executor.map(full_content, urls))

# Function to process the data
def process_data(df):
    # Fetch content for all URLs
    df['content'] = fetch_content_parallel(df['url'])
    
    # Clean the content by replacing newline characters with spaces
    df['content'] = df['content'].str.replace('\n', ' ')
    
    # Remove rows where content could not be retrieved
    df = df[df['content'] != 'couldnt retrieve']

    # Count words in the content using spaCy
    def count_words(text):
        doc = nlp(str(text))
        return len([token.text for token in doc if not token.is_stop])

    df['word_count'] = df['content'].apply(count_words)
    
    # Initialize SentimentIntensityAnalyzer
    sid = SentimentIntensityAnalyzer()

    # Function to get sentiment of the text
    def get_sentiment(row):
        scores = sid.polarity_scores(row)
        return ('Positive' if scores['compound'] >= 0.05 else 'Negative' if scores['compound'] <= -0.05 else 'Neutral', scores['compound'])
    
    df[['sentiment', 'compound_score']] = df['content'].astype(str).apply(lambda x: pd.Series(get_sentiment(x)))

    return df

# Process the data
dataframe = process_data(dataframe)

# Save processed data in Delta format
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("NewsProcessing").getOrCreate()
spark_df = spark.createDataFrame(dataframe)
spark_df.write.format("delta").mode('overwrite').save("/mnt/data/processed_news")


Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
Python interpreter will be restarted.
2025-04-02 21:27:33.083516: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
    

*** WARNING: max output size exceeded, skipping output. ***

     |████████████████████████████████| 12.8 MB 6.7 MB/s 
You should consider upgrading via the '/local_disk0/.ephemeral_nfs/envs/pythonEnv-77d6ca89-c120-4cf1-98fb-f30c44f00e18/bin/python -m pip install --upgrade pip' command.
✔ Download and installation successful
You can now load the 

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
ERROR:__main__:Error retrieving content from https://parade.com/news/denise-richards-hard-confession-charlie-sheen-health-condition: Article `download()` failed with 403 Client Error: Forbidden for url: https://parade.com/news/denise-richards-hard-confession-charlie-sheen-health-condition on URL https://parade.com/news/denise-richards-hard-confession-charlie-sheen-health-condition
ERROR:__main__:Error retrieving content from https://www.chron.com/culture/tv/article/profanity-texas-film-incentive-20250234.php: Article `download()` failed with 403 Client Error: Forbidden for url: https://www.chron.com/culture/tv/article/profanity-texas-film-incentive-20250234.php on URL https://www.chron.com/culture/tv/article/profanity-texas-film-incentive-20250234.php
ERROR:__main__:Error retrieving content from https://www.gamespot.com/articles/the-first-social-media-reviews-to-a-minecraft-movie-are-in-heres-what-people-are-saying/110